# 🔄 Query Transformation

**Day 8 — Notebook 2 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Why the user's raw query is often suboptimal for retrieval
- Query rewriting: clarifying and expanding vague questions
- HyDE (Hypothetical Document Embeddings): a counter-intuitive but powerful technique
- Multi-query retrieval: generating multiple search angles for broader recall
- Step-back prompting: abstracting specific questions to improve context retrieval

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb langchain-text-splitters

In [ ]:
import sys, os, hashlib
import numpy as np
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
EMBEDDING_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-2.5-flash"

---

## 1. Why Raw Queries Are Suboptimal

Users write queries the way they would type in a search box — short, ambiguous, sometimes containing typos or colloquialisms. These raw queries often don't match the language used in the documents.

```
User query:  "how do i fix slow python code"
Document:    "Techniques for Python performance optimisation and profiling"

User query:  "what's RAG?"
Document:    "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"

User query:  "best way to store vectors"
Document:    "Vector database indexing strategies: HNSW, IVF, and Flat indexes"
```

Query transformation bridges this gap by rewriting or expanding the query before retrieval.

In [ ]:
# Build a knowledge base for experiments
KNOWLEDGE_BASE = [
    {"id": "perf-1", "text": "Python performance optimisation: use NumPy for numerical operations instead of pure Python loops. Profile code with cProfile or line_profiler to identify bottlenecks before optimising."},
    {"id": "perf-2", "text": "Caching with functools.lru_cache can dramatically speed up recursive or repeated function calls in Python. Use @lru_cache(maxsize=None) for unlimited caching."},
    {"id": "rag-1",  "text": "Retrieval-Augmented Generation (RAG) is an AI architecture that retrieves relevant documents from a knowledge base and uses them as context for a language model to generate grounded answers."},
    {"id": "rag-2",  "text": "Advanced RAG techniques include query rewriting, HyDE (Hypothetical Document Embeddings), hybrid retrieval, and re-ranking to improve the quality of retrieved context."},
    {"id": "vec-1",  "text": "Vector databases store high-dimensional embeddings and support approximate nearest neighbour (ANN) search. Popular options include ChromaDB, Pinecone, Weaviate, and Qdrant."},
    {"id": "vec-2",  "text": "HNSW (Hierarchical Navigable Small World) is the most widely used ANN index. It builds a layered graph structure that enables logarithmic-time search."},
    {"id": "llm-1",  "text": "Large language models (LLMs) are neural networks trained on vast amounts of text data. They predict the next token based on the context window, generating coherent and contextually appropriate text."},
    {"id": "llm-2",  "text": "Hallucinations in LLMs occur when the model generates factually incorrect information with high confidence. Grounding techniques like RAG help reduce hallucinations by anchoring answers to retrieved evidence."},
    {"id": "emb-1",  "text": "Text embeddings are dense vector representations of text that capture semantic meaning. The Gemini gemini-embedding-001 model produces 3072-dimensional embeddings optimised for retrieval tasks."},
    {"id": "emb-2",  "text": "Cosine similarity measures the angle between two embedding vectors, regardless of magnitude. A score of 1.0 means identical direction; 0.0 means orthogonal (unrelated topics)."},
]

# Build ChromaDB index
chroma = chromadb.Client()
col = chroma.create_collection("query_transform", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in KNOWLEDGE_BASE]
vecs = client.models.embed_content(
    model=EMBEDDING_MODEL, contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
).embeddings

col.add(
    ids=[d["id"] for d in KNOWLEDGE_BASE],
    documents=texts,
    embeddings=[e.values for e in vecs],
)
print(f"✅ Index built: {col.count()} documents")


def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Base dense retrieval — used to compare transformation techniques."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL, contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values
    results = col.query(query_embeddings=[q_vec], n_results=top_k,
                        include=["documents", "distances"])
    return [
        {"id": results["ids"][0][i], "text": results["documents"][0][i],
         "score": 1 - results["distances"][0][i]}
        for i in range(len(results["ids"][0]))
    ]

---

## 2. Query Rewriting

Use the LLM to rewrite the user's informal query into a more precise, retrieval-optimised form.

In [ ]:
def rewrite_query(original_query: str) -> str:
    """Rewrite a user query to be more precise and retrieval-friendly."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Rewrite the following user question to be more specific, precise, and 
retrieval-friendly for a technical knowledge base. Expand abbreviations, clarify vague terms, 
and use technical vocabulary. Return ONLY the rewritten query with no explanation.

Original query: {original_query}
Rewritten query:""",
        config=types.GenerateContentConfig(temperature=0.0),
    )
    return response.text.strip()


# Test rewriting
raw_queries = [
    "how do i fix slow python code",
    "what's RAG?",
    "best way to store vectors",
    "why does my llm make stuff up",
]

print("Query Rewriting Results:\n")
for q in raw_queries:
    rewritten = rewrite_query(q)
    print(f"Original:  {q}")
    print(f"Rewritten: {rewritten}")

    # Compare retrieval
    orig_top = retrieve(q, top_k=1)
    rewr_top = retrieve(rewritten, top_k=1)
    print(f"Original  top-1: [{orig_top[0]['score']:.3f}] {orig_top[0]['text'][:60]}...")
    print(f"Rewritten top-1: [{rewr_top[0]['score']:.3f}] {rewr_top[0]['text'][:60]}...")
    print()

---

## 3. HyDE — Hypothetical Document Embeddings

**HyDE** is a clever technique: instead of embedding the query directly, ask the LLM to *generate a hypothetical answer*, then embed that.

```
Normal RAG:   embed(query)          → search → retrieve documents
HyDE:         embed(hypothetical_answer) → search → retrieve documents
```

**Why it works:** A hypothetical answer lives in the same embedding space as real documents. A generated answer "sounds like" a document paragraph — so it's closer in embedding space to the actual relevant document than the original short query.

In [ ]:
def hyde_retrieve(query: str, top_k: int = 3) -> tuple[list[dict], str]:
    """HyDE retrieval: generate a hypothetical answer, embed it, then search."""
    # Step 1: Generate a hypothetical answer
    hypo_response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Write a brief, factual paragraph (3-4 sentences) that would directly answer 
this question. Write as if it is an excerpt from a technical documentation page.

Question: {query}
Hypothetical answer:""",
        config=types.GenerateContentConfig(temperature=0.2),
    )
    hypothetical = hypo_response.text.strip()

    # Step 2: Embed the hypothetical answer (not the query)
    h_vec = client.models.embed_content(
        model=EMBEDDING_MODEL, contents=hypothetical,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    ).embeddings[0].values

    # Step 3: Search with the hypothetical embedding
    results = col.query(query_embeddings=[h_vec], n_results=top_k,
                        include=["documents", "distances"])
    chunks = [
        {"id": results["ids"][0][i], "text": results["documents"][0][i],
         "score": 1 - results["distances"][0][i]}
        for i in range(len(results["ids"][0]))
    ]
    return chunks, hypothetical


# Compare HyDE vs direct retrieval
test_queries = [
    "how do i fix slow python code",
    "what causes language models to invent facts",
]

for q in test_queries:
    direct = retrieve(q, top_k=2)
    hyde_chunks, hypothetical = hyde_retrieve(q, top_k=2)

    print(f"❓ Query: '{q}'")
    print(f"\n💭 Hypothetical document (HyDE):")
    print(f"   {hypothetical[:200]}...")
    print(f"\n📊 Direct retrieval top-2:")
    for r in direct:
        print(f"   [{r['score']:.3f}] {r['text'][:80]}...")
    print(f"\n📊 HyDE retrieval top-2:")
    for r in hyde_chunks:
        print(f"   [{r['score']:.3f}] {r['text'][:80]}...")
    print("\n" + "=" * 60 + "\n")

---

## 4. Multi-Query Retrieval

Generate **multiple rephrased versions** of the query and retrieve for each. Then deduplicate and merge. This improves recall for complex questions that have multiple valid search angles.

In [ ]:
from pydantic import BaseModel

class MultiQuery(BaseModel):
    queries: list[str]


def generate_multi_query(original_query: str, n: int = 3) -> list[str]:
    """Generate n alternative phrasings of the query."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Generate {n} different ways to phrase this question for a search engine.
Each phrasing should capture a different angle or use different vocabulary.
Return as a JSON object with a 'queries' list.

Original question: {original_query}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=MultiQuery,
            temperature=0.7,
        ),
    )
    result = MultiQuery.model_validate_json(response.text)
    return result.queries


def multi_query_retrieve(original_query: str, top_k: int = 3) -> list[dict]:
    """Retrieve using multiple query phrasings and deduplicate."""
    all_queries = [original_query] + generate_multi_query(original_query, n=3)

    seen_ids = set()
    all_results = []

    for q in all_queries:
        for r in retrieve(q, top_k=top_k):
            if r["id"] not in seen_ids:
                seen_ids.add(r["id"])
                all_results.append(r)

    # Sort by best score and return top_k
    all_results.sort(key=lambda x: x["score"], reverse=True)
    return all_results[:top_k]


# Demonstrate multi-query
original = "How do vector databases speed up search?"
variants = generate_multi_query(original, n=3)

print(f"Original: '{original}'")
print("\nGenerated variants:")
for i, v in enumerate(variants, 1):
    print(f"  {i}. {v}")

print("\nMulti-query top-3 results:")
results = multi_query_retrieve(original, top_k=3)
for r in results:
    print(f"  [{r['score']:.3f}] {r['text'][:80]}...")

---

## 5. Step-Back Prompting

For very specific questions, retrieve broader context first by asking a more **general "step-back" question**, then combine both retrieval results.

In [ ]:
def step_back_retrieve(specific_query: str, top_k: int = 3) -> list[dict]:
    """Retrieve both specific and broader context using step-back."""
    # Generate a more general version of the question
    stepback_response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Given this specific question, write a more general, broader version that would 
retrieve useful background context. Return ONLY the broader question.

Specific question: {specific_query}
Broader question:""",
        config=types.GenerateContentConfig(temperature=0.0),
    )
    broader = stepback_response.text.strip()

    # Retrieve for both specific and broad
    specific_results = retrieve(specific_query, top_k=top_k)
    broad_results = retrieve(broader, top_k=top_k)

    # Deduplicate and merge
    seen = set()
    combined = []
    for r in specific_results + broad_results:
        if r["id"] not in seen:
            seen.add(r["id"])
            combined.append(r)

    print(f"  Step-back query: '{broader}'")
    return combined[:top_k]


specific_questions = [
    "What is the maxsize parameter in lru_cache?",
    "What is the score range for cosine similarity?",
]

for q in specific_questions:
    print(f"\n❓ Specific: '{q}'")
    results = step_back_retrieve(q, top_k=2)
    print("  Retrieved:")
    for r in results:
        print(f"    [{r['score']:.3f}] {r['text'][:80]}...")

---

## 6. Putting It Together: Adaptive Query Pipeline

In [ ]:
def adaptive_rag(question: str, strategy: str = "rewrite") -> str:
    """
    RAG with selectable query transformation strategy.
    strategy: 'direct' | 'rewrite' | 'hyde' | 'multi_query'
    """
    if strategy == "direct":
        chunks = retrieve(question, top_k=3)
    elif strategy == "rewrite":
        rewritten = rewrite_query(question)
        chunks = retrieve(rewritten, top_k=3)
    elif strategy == "hyde":
        chunks, _ = hyde_retrieve(question, top_k=3)
    elif strategy == "multi_query":
        chunks = multi_query_retrieve(question, top_k=3)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    context = "\n\n".join(f"[{c['id']}] {c['text']}" for c in chunks)
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"Answer using only the context. Cite document IDs.\n\n<context>\n{context}\n</context>\n\nQuestion: {question}",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


q = "why does my llm make stuff up and how do i fix it"
print(f"Question: '{q}'\n")

for strategy in ["direct", "rewrite", "hyde"]:
    answer = adaptive_rag(q, strategy=strategy)
    print(f"[{strategy.upper()}] {answer[:200]}...\n")

---

## 🏋️ Exercise 1: Query Rewriting Evaluation

Measure whether query rewriting improves retrieval recall on a test set.

In [ ]:
# Exercise 1: Evaluate query rewriting
eval_set = [
    ("how do vectors capture meaning",      "emb-1"),
    ("speed up python",                     "perf-1"),
    ("what is rag in ai",                   "rag-1"),
    ("how similar are two texts",           "emb-2"),
    ("llm making things up",                "llm-2"),
]

# TODO:
# 1. For each (query, expected_id), retrieve top-3 with and without rewriting
# 2. Check if expected_id appears in the results
# 3. Print: Recall@3 with direct retrieval vs with query rewriting

direct_hits = 0
rewrite_hits = 0

for query, expected_id in eval_set:
    pass  # TODO

# print(f"Direct recall@3:  {direct_hits/len(eval_set):.0%}")
# print(f"Rewrite recall@3: {rewrite_hits/len(eval_set):.0%}")

---

## 🏋️ Exercise 2: Multi-Query with Deduplication

How many unique documents does multi-query retrieve compared to single-query?

In [ ]:
# Exercise 2: Multi-query coverage analysis
# TODO:
# For this complex question, compare:
# - Single query: how many unique doc IDs in top-3?
# - Multi-query: how many unique doc IDs are retrieved across all variants (before dedup)?
# - Multi-query deduped top-3: are different docs retrieved than single query?

complex_q = "What are the key techniques for building a high-quality RAG system that avoids hallucinations?"

# TODO: Implement and print the comparison

---

## Key Takeaways

| Technique | When to Use | Cost |
|-----------|-------------|------|
| **Query rewriting** | Informal or ambiguous queries | 1 extra LLM call |
| **HyDE** | Short queries with vocabulary mismatch | 1 extra LLM call |
| **Multi-query** | Complex questions with multiple angles | N extra LLM calls |
| **Step-back** | Very specific questions needing context | 1 extra LLM call + extra retrieval |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| HyDE Paper | Paper | [arxiv.org/abs/2212.10496](https://arxiv.org/abs/2212.10496) |
| Step-Back Prompting | Paper | [arxiv.org/abs/2310.06117](https://arxiv.org/abs/2310.06117) |
| LangChain MultiQueryRetriever | Docs | [python.langchain.com/docs/modules/data_connection/retrievers/MultiQueryRetriever](https://python.langchain.com/docs/modules/data_connection/retrievers/MultiQueryRetriever/) |

---

**Next up:** [03_Reranking_Models.ipynb](./03_Reranking_Models.ipynb)